### Validate different types of errors using MAP as an example

#### Invalid values or out of range

In the following example, several fields are mutated to have invalid values or with values out of range. The validator collects all errors and pinpoint the exact field that is invalid, with nested key for dictionary and indexing for list type.

In [1]:
import copy
from pprint import pprint
import yaml
from v2x_msg_validator.validator import V2XMessageValidator


def read_sample_yaml(filepath):
    with open(filepath, "r") as file:
        data = yaml.safe_load(file)
    return data

data = read_sample_yaml("../test/j2735_202409/fixtures/MAP.yaml")
pprint(data)

{'messageId': 18,
 'value': {'MapData': {'dataParameters': {'geoidUsed': 'WGS84',
                                          'lastCheckedDate': '2024-01-15',
                                          'processAgency': 'TestAgency',
                                          'processMethod': 'RTCM'},
                       'intersections': [{'id': {'id': 1001, 'region': 100},
                                          'laneSet': [{'connectsTo': [{'connectingLane': {'lane': 25,
                                                                                          'maneuver': '8000'},
                                                                       'connectionID': 10,
                                                                       'remoteIntersection': {'id': 1002,
                                                                                              'region': 100},
                                                                       'signalGroup': 2,
                

In [7]:
# mutate data
_data = copy.deepcopy(data)

# duplicate the intersection list item to showcase indexing
_data["value"]["MapData"]["intersections"].append(copy.deepcopy(_data["value"]["MapData"]["intersections"][0]))

_data["value"]["MapData"]["msgIssueRevision"] = 128 # out of range
_data["value"]["MapData"]["intersections"][0]["refPoint"]["lat"] = "not_a_number"

# modify the added list
_data["value"]["MapData"]["intersections"][1]["laneSet"][0]["nodeList"]["nodes"][0]["delta"]["node-XY6"]["y"] = "bad"

validator = V2XMessageValidator(revision="j2735_202409")
errors = validator.validate(_data)

pprint(errors)

[ASN1ObjValErr[key='MapData.intersections[0].refPoint.lat', code=0, msg='invalid named value', val='not_a_number'],
 ASN1ObjValErr[key='MapData.intersections[1].laneSet[0].nodeList.nodes[0].delta.node-XY6.y', code=0, msg='invalid named value', val='bad'],
 ASN1ObjValErr[key='MapData.msgIssueRevision', code=3, msg='INTEGER value out of constraint', val=128]]


#### Missing keys

A required key is deleted: `msgIssueRevision`.

In [8]:
_data = copy.deepcopy(data)
del _data["value"]["MapData"]["msgIssueRevision"]

errors = validator.validate(_data)
pprint(errors)

[ASN1ObjValErr[key='MapData', code=1, msg="missing mandatory value(s): {'msgIssueRevision'}", val={'timeStamp': 400000, 'layerType': 'intersectionData', 'layerID': 1, 'intersections': [{'name': 'TestIntersection', 'id': {'region': 100, 'id': 1001}, 'revision': 3, 'refPoint': {'lat': 389284111, 'long': -772410713, 'elevation': 1000}, 'laneWidth': 366, 'speedLimits': [{'type': 'maxSpeedInSchoolZone', 'speed': 500}], 'laneSet': [{'laneID': 1, 'name': 'Lane1', 'ingressApproach': 1, 'egressApproach': 2, 'laneAttributes': {'directionalUse': '10', 'sharedWith': '0000', 'laneType': {'vehicle': '00'}}, 'maneuvers': '8000', 'nodeList': {'nodes': [{'delta': {'node-XY6': {'x': -4012, 'y': 365}}, 'attributes': {'localNode': ['stopLine'], 'disabled': ['doNotBlock'], 'enabled': ['adaptiveTimingPresent'], 'data': [{'pathEndPointAngle': 10}], 'dWidth': 50, 'dElevation': 10}}, {'delta': {'node-XY6': {'x': -5541, 'y': 7249}}}]}, 'connectsTo': [{'connectingLane': {'lane': 25, 'maneuver': '8000'}, 'remoteI

#### Additional key

An `extra_key` is added which does not belong in the schema.

In [9]:
_data = copy.deepcopy(data)
_data["value"]["MapData"]["extra_key"] = "some_value"
errors = validator.validate(_data)
pprint(errors)

[ASN1ObjValErr[key='MapData', code=2, msg='invalid key: extra_key', val={'timeStamp': 400000, 'msgIssueRevision': 5, 'layerType': 'intersectionData', 'layerID': 1, 'intersections': [{'name': 'TestIntersection', 'id': {'region': 100, 'id': 1001}, 'revision': 3, 'refPoint': {'lat': 389284111, 'long': -772410713, 'elevation': 1000}, 'laneWidth': 366, 'speedLimits': [{'type': 'maxSpeedInSchoolZone', 'speed': 500}], 'laneSet': [{'laneID': 1, 'name': 'Lane1', 'ingressApproach': 1, 'egressApproach': 2, 'laneAttributes': {'directionalUse': '10', 'sharedWith': '0000', 'laneType': {'vehicle': '00'}}, 'maneuvers': '8000', 'nodeList': {'nodes': [{'delta': {'node-XY6': {'x': -4012, 'y': 365}}, 'attributes': {'localNode': ['stopLine'], 'disabled': ['doNotBlock'], 'enabled': ['adaptiveTimingPresent'], 'data': [{'pathEndPointAngle': 10}], 'dWidth': 50, 'dElevation': 10}}, {'delta': {'node-XY6': {'x': -5541, 'y': 7249}}}]}, 'connectsTo': [{'connectingLane': {'lane': 25, 'maneuver': '8000'}, 'remoteInte

### Validate SPaT message

Validating different message types share the same API.

In [5]:
data = read_sample_yaml("../test/j2735_202409/fixtures/SPAT.yaml")
pprint(data)

{'messageId': 19,
 'value': {'SPAT': {'intersections': [{'enabledLanes': [1, 2],
                                       'id': {'id': 9709, 'region': 100},
                                       'maneuverAssistList': [{'availableStorageLength': 150,
                                                               'connectionID': 10,
                                                               'pedBicycleDetect': True,
                                                               'queueLength': 50,
                                                               'waitOnStop': False}],
                                       'moy': 428788,
                                       'name': 'West Intersection',
                                       'revision': 1,
                                       'states': [{'maneuverAssistList': [{'availableStorageLength': 200,
                                                                           'connectionID': 5,
                                   

In [6]:
errors = validator.validate(data)
pprint(errors)

[]


### Validate TravelerInformation message

In [7]:
data = read_sample_yaml("../test/j2735_202409/fixtures/TIM.yaml")
pprint(data)

{'messageId': 31,
 'value': {'TravelerInformation': {'dataFrames': [{'content': {'speedLimit': [{'item': {'itis': 27}},
                                                                              {'item': {'text': 'Speed '
                                                                                                '25'}}]},
                                                   'doNotUse1': 0,
                                                   'doNotUse2': 0,
                                                   'doNotUse3': 0,
                                                   'doNotUse4': 0,
                                                   'durationTime': 32000,
                                                   'frameType': 'advisory',
                                                   'msgId': {'roadSignID': {'crc': '0000',
                                                                            'mutcdCode': 'warning',
                                                            

In [8]:
errors = validator.validate(data)
pprint(errors)

[]


The validated message is encoded with UPER and subsequently decoded, which match the initial data.

In [9]:
import json

from v2x_codecs import j2735_202409

TIM = j2735_202409.TravelerInformation.TravelerInformation
# convert to uper encoding, decode, and compare against original data
buf = TIM.to_uper()

print("==== uper encoding ===")
pprint(buf)

# decode from buffer
TIM.from_uper(buf)

# convert to json
data_decoded = json.loads(TIM.to_json())

print("==== decoded json ===")
pprint(data_decoded)


assert data["value"]["TravelerInformation"]==data_decoded
print("Decoded data match initial data")

==== uper encoding ===
(b'p\x16\x05N\x00\x00\x00\x00\x00\x00\x00\x00\x00GGN\x9c\x1d/_\x97\xc6\x1d\xbc'
 b'6e]\x8f~\xd0\xc0z\x99\xb9\xea\xc2z\x9b\xaat#!X @\x00\x0f\xd0\xc0\xa9\xdf@'
 b'(\x01~&\xa6^})e\xcf\xa7~\xe53s\xd5\x84\xf57T\xe8F@\x0br\x80\x00\x00'
 b'\x01`\xfa!\xf4\x0b\x0b\xb9\x13\x88\x001\x00\r\xde\x9f\x0c\xb9r d\xd6\xb4t'
 b'\xe9\xc1\xd2\xf5\xfe\x17c\xde')
==== decoded json ===
{'dataFrames': [{'content': {'speedLimit': [{'item': {'itis': 27}},
                                            {'item': {'text': 'Speed 25'}}]},
                 'doNotUse1': 0,
                 'doNotUse2': 0,
                 'doNotUse3': 0,
                 'doNotUse4': 0,
                 'durationTime': 32000,
                 'frameType': 'advisory',
                 'msgId': {'roadSignID': {'crc': '0000',
                                          'mutcdCode': 'warning',
                                          'position': {'elevation': 400,
                                                       '

### Validate PersonalSafetyMessage

In [10]:
data = read_sample_yaml("../test/j2735_202409/fixtures/PSM.yaml")
pprint(data)

{'messageId': 32,
 'value': {'PersonalSafetyMessage': {'accelSet': {'lat': 0,
                                                  'long': 0,
                                                  'vert': 0,
                                                  'yaw': 0},
                                     'accuracy': {'orientation': 0,
                                                  'semiMajor': 50,
                                                  'semiMinor': 50},
                                     'activitySubType': '00',
                                     'activityType': '00',
                                     'animalType': 'pet',
                                     'assistType': '00',
                                     'attachment': 'stroller',
                                     'attachmentRadius': 50,
                                     'basicType': 'aPEDESTRIAN',
                                     'clusterRadius': 10,
                                     'clusterSize': '

In [11]:
errors = validator.validate(data)
pprint(errors)

[]


In [12]:
from v2x_codecs import j2735_202409

PSM = j2735_202409.PersonalSafetyMessage.PersonalSafetyMessage
# convert to uper encoding, decode, and compare against original data
buf = PSM.to_uper()

print("==== uper encoding ===")
pprint(buf)

# decode from buffer
PSM.from_uper(buf)

# convert to json
data_decoded = json.loads(PSM.to_json())

print("==== decoded json ===")
pprint(data_decoded)

assert data["value"]["PersonalSafetyMessage"]==data_decoded
print("Decoded data match initial data")

==== uper encoding ===
(b'_\xff\xc3_\x90\x04\x00\x00\x00\x05L\xdc\xf5a=M\xd5:\x11\x9022\x00\x00'
 b'\x03\xe9\xf4\x07\xd0}\x07\xf7\xff\xf7\xff\xf6@ \x02"\x90\x00\x00\x00'
 b'\x04\xc9\x00')
==== decoded json ===
{'accelSet': {'lat': 0, 'long': 0, 'vert': 0, 'yaw': 0},
 'accuracy': {'orientation': 0, 'semiMajor': 50, 'semiMinor': 50},
 'activitySubType': '00',
 'activityType': '00',
 'animalType': 'pet',
 'assistType': '00',
 'attachment': 'stroller',
 'attachmentRadius': 50,
 'basicType': 'aPEDESTRIAN',
 'clusterRadius': 10,
 'clusterSize': 'small',
 'crossRequest': True,
 'crossState': False,
 'eventResponderType': 'lawEnforcement',
 'heading': 8000,
 'id': '00000001',
 'msgCnt': 1,
 'pathPrediction': {'confidence': 200, 'radiusOfCurve': 32767},
 'position': {'elevation': 400, 'lat': 389549153, 'long': -771488965},
 'propulsion': {'human': 'onFoot'},
 'secMark': 45000,
 'sizing': '00',
 'speed': 125,
 'useState': '0000'}
Decoded data match initial data
